# 12 - V2.4 Tail Recovery Lab

Objectif:
- corriger la **sous-modélisation de la queue** sur la sévérité,
- préserver le RMSE global et la stabilité inter-splits,
- produire 2 soumissions (`robust`, `lb_challenger`) via des corrections **tail-only**.

Tous les outputs sont écrits dans `artifacts/v2_4_tail_recovery/`.


## Rappel méthodologique

- On ne corrige pas la queue avec un simple scaling global.
- On applique des corrections **au-dessus d'un seuil** (tail-only).
- On sélectionne via un compromis **RMSE + queue + stabilité**, pas via `q99=1.0`.


In [1]:
import sys
import json
import itertools
from pathlib import Path
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")

NOTEBOOK_DIR = Path.cwd().resolve()
ROOT = NOTEBOOK_DIR
for _candidate in [NOTEBOOK_DIR, *NOTEBOOK_DIR.parents]:
    if (_candidate / "src").exists() and (_candidate / "data").exists():
        ROOT = _candidate
        break
else:
    raise RuntimeError("Project root not found. Expected folders: src/ and data/.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
ARTIFACTS_DIR = ROOT / "artifacts"

from src.insurance_pricing.experiments.quick import tail_recovery as tr

# =========================
# Config V2.4
# =========================
RUN_PHASE_A1 = True
RUN_PHASE_A2 = True
RUN_PHASE_B1 = False         # optional heavy retrain
ALLOW_PHASE_B_SELECTION = False
QUICK_MODE = True

SEED = 42
ARTIFACT_V24 = tr.ensure_v24_dir(ROOT)

THRESHOLD_GRID_A1 = [0.90, 0.95, 0.975]
LAMBDA_GRID_A1 = [0.10, 0.20, 0.35, 0.50]
GAMMA_GRID_A1 = [1.0, 2.0, 3.0]

THRESHOLD_GRID_A2 = [0.90, 0.95]
MAPPER_MODES_A2 = ["piecewise_monotone", "isotonic_safe"]
MIN_SAMPLES_TAIL_A2 = [150, 250]

# Optional Phase B (weighted-tail retrain)
PHASE_B_SEV_PARAM_VARIANTS = [
    {},  # default weighted_tail
]

print("ROOT:", ROOT)
print("ARTIFACT_V24:", ARTIFACT_V24)


ROOT: c:\Users\icemo\Downloads\Calcul-prime-d-assurance
ARTIFACT_V24: c:\Users\icemo\Downloads\Calcul-prime-d-assurance\artifacts\v2_4_tail_recovery


## 1) Chargement des artefacts V1/V2/V2.2/V2.3 et extraction de l'ancre robuste


In [13]:
ctx = tr.load_tail_recovery_context(ROOT)
base = tr.extract_base_run_predictions(ctx, run_id=None, split="primary_time")

print("Base run id:", base["base_run_id"])
print("Base run row keys:", sorted(list(base["base_run_row"].keys()))[:20], "...")
print("Splits available:", {k: v.shape for k, v in base["pred_by_split"].items()})

base_ds_diag = base.get("ds_diag", {})
display(base_ds_diag.get("metrics", pd.DataFrame()))
display(base_ds_diag.get("error_by_decile_true_zero_aware", pd.DataFrame()).head(15))
display(base_ds_diag.get("error_by_decile_pos_only", pd.DataFrame()).head(15))


Base run id: base_v2|catboost|two_part_tweedie|cb_v2_c1|42|weighted_tail|isotonic|isotonic
Base run row keys: ['auc_freq', 'brier_freq', 'calibration', 'config_id', 'engine', 'family', 'feature_set', 'fold_id', 'level', 'n_valid', 'q99_ratio_pos', 'rmse_prime', 'rmse_sev_pos', 'run_id', 'seed', 'severity_mode', 'split', 'tail_mapper'] ...
Splits available: {'primary_time': (100000, 18), 'secondary_group': (100000, 18), 'aux_blocked5': (100000, 18)}


,run_id,split,n,n_nonzero,share_nonzero,rmse_prime,mae_prime,r2_prime,auc_freq,gini_freq,brier_freq,logloss_freq,pr_auc_freq,rmse_sev_pos,mae_sev_pos,q95_ratio_pos,q99_ratio_pos,mae_prime_nonzero,r2_prime_nonzero,rmse_prime_top1pct
0,base_v2|catboost|two_part_tweedie|cb_v2_c1|42|...,primary_time,50000,2917,0.05834,545.929482,174.134052,-0.001952,0.594651,0.189301,0.054613,0.21989,0.075693,1662.270082,1106.900817,0.476975,0.32863,1675.786354,-1.308247,4548.227515


,decile_basis,bin_type,bin_order,bin,n,y_mean,pred_mean,bias,mae,rmse
0,y_true_zero_aware,zero,0,zero,47083,0.000000,81.100053,81.100053,81.100053,100.615752
1,y_true_zero_aware,positive_decile,1,"(600.599, 698.79]",292,644.527500,99.766043,-544.761457,544.930072,550.277253
2,y_true_zero_aware,positive_decile,2,"(698.79, 851.978]",292,772.985342,96.150867,-676.834476,676.834476,680.605112
3,y_true_zero_aware,positive_decile,3,"(851.978, 1010.41]",331,957.972085,88.952121,-869.019963,869.019963,873.282006
4,y_true_zero_aware,positive_decile,4,"(1010.41, 1224.192]",252,1114.050952,98.798698,-1015.252255,1015.252255,1018.862386
5,y_true_zero_aware,positive_decile,5,"(1224.192, 1236.0]",346,1235.759364,75.905501,-1159.853863,1159.853863,1161.232854
6,y_true_zero_aware,positive_decile,6,"(1236.0, 1442.87]",259,1337.212664,100.502512,-1236.710152,1236.710152,1239.942841
7,y_true_zero_aware,positive_decile,7,"(1442.87, 1890.012]",270,1650.654889,97.246145,-1553.408744,1553.408744,1560.494063
8,y_true_zero_aware,positive_decile,8,"(1890.012, 2487.526]",291,2180.740722,102.991911,-2077.748810,2077.748810,2087.066466
9,y_true_zero_aware,positive_decile,9,"(2487.526, 3336.74]",292,2812.897260,102.709318,-2710.187943,2710.187943,2721.394782


,decile_basis,bin_type,bin_order,bin,n,y_mean,pred_mean,bias,mae,rmse
0,y_true_positive_only,qcut_all,1,"(1010.41, 1224.192]",252,1114.050952,98.798698,-1015.252255,1015.252255,1018.862386
1,y_true_positive_only,qcut_all,2,"(1224.192, 1236.0]",346,1235.759364,75.905501,-1159.853863,1159.853863,1161.232854
2,y_true_positive_only,qcut_all,3,"(1236.0, 1442.87]",259,1337.212664,100.502512,-1236.710152,1236.710152,1239.942841
3,y_true_positive_only,qcut_all,4,"(1442.87, 1890.012]",270,1650.654889,97.246145,-1553.408744,1553.408744,1560.494063
4,y_true_positive_only,qcut_all,5,"(1890.012, 2487.526]",291,2180.740722,102.991911,-2077.748810,2077.748810,2087.066466
5,y_true_positive_only,qcut_all,6,"(2487.526, 3336.74]",292,2812.897260,102.709318,-2710.187943,2710.187943,2721.394782
6,y_true_positive_only,qcut_all,7,"(3336.74, 21826.96]",292,5072.515959,103.382877,-4969.133082,4969.133082,5469.479943
7,y_true_positive_only,qcut_all,8,"(600.599, 698.79]",292,644.527500,99.766043,-544.761457,544.930072,550.277253
8,y_true_positive_only,qcut_all,9,"(698.79, 851.978]",292,772.985342,96.150867,-676.834476,676.834476,680.605112
9,y_true_positive_only,qcut_all,10,"(851.978, 1010.41]",331,957.972085,88.952121,-869.019963,869.019963,873.282006


In [14]:
# Base diagnostics summary (primary_time)
base_primary = base["pred_by_split"].get("primary_time", pd.DataFrame()).copy()
base_primary_oof = base_primary[base_primary["is_test"] == 0].copy()
base_primary_test = base_primary[base_primary["is_test"] == 1].copy()

base_summary = {}
if len(base_primary_oof):
    y = base_primary_oof["y_sev"].to_numpy(dtype=float)
    p = base_primary_oof["pred_prime"].to_numpy(dtype=float)
    pos = y > 0
    base_summary = {
        "n": int(len(y)),
        "n_nonzero": int(np.sum(pos)),
        "rmse_prime_primary": float(np.sqrt(np.mean((y - p) ** 2))),
        "q95_true_pos": float(np.quantile(y[pos], 0.95)) if np.any(pos) else np.nan,
        "q95_predsev_pos": float(np.quantile(base_primary_oof.loc[pos, 'pred_sev'].to_numpy(dtype=float), 0.95)) if np.any(pos) else np.nan,
        "q99_true_pos": float(np.quantile(y[pos], 0.99)) if np.any(pos) else np.nan,
        "q99_predsev_pos": float(np.quantile(base_primary_oof.loc[pos, 'pred_sev'].to_numpy(dtype=float), 0.99)) if np.any(pos) else np.nan,
    }
    if np.any(pos):
        base_summary["q95_ratio_pos"] = base_summary["q95_predsev_pos"] / max(base_summary["q95_true_pos"], 1e-9)
        base_summary["q99_ratio_pos"] = base_summary["q99_predsev_pos"] / max(base_summary["q99_true_pos"], 1e-9)
display(pd.DataFrame([base_summary]))


,n,n_nonzero,rmse_prime_primary,q95_true_pos,q95_predsev_pos,q99_true_pos,q99_predsev_pos,q95_ratio_pos,q99_ratio_pos
0,50000,2917,545.929482,4355.258,2077.348952,6963.1188,2288.287593,0.476975,0.32863


## 2) Phase A1 — Tail rank scaling (tail-only, sans réentraînement)

Forme:
- sous le seuil: identité
- au-dessus: multiplicateur croissant selon le rang empirique


In [15]:
transform_store = {}
candidate_results = []

# baseline identity candidate for comparison
baseline_identity = tr.score_tail_candidate_multi_split(
    candidate_id="baseline_identity",
    base_run_id=base["base_run_id"],
    candidate_family="baseline_identity",
    pred_df_all_splits=base["pred_all"],
    transform_kind="identity",
    transform_params={"kind": "identity"},
)
candidate_results.append(baseline_identity)
transform_store["baseline_identity"] = {"transform_kind": "identity", "transform_params": {"kind": "identity"}}

if RUN_PHASE_A1:
    base_primary_oof_pos = base_primary_oof[base_primary_oof["y_sev"] > 0].copy()
    grid_a1 = list(itertools.product(THRESHOLD_GRID_A1, LAMBDA_GRID_A1, GAMMA_GRID_A1))
    if QUICK_MODE:
        grid_a1 = grid_a1[:18]  # quick cap

    print("A1 grid size:", len(grid_a1))
    for threshold_q, lambda_, gamma_ in grid_a1:
        params = tr.fit_tail_rank_scaler(
            base_primary_oof,
            threshold_q=float(threshold_q),
            lambda_=float(lambda_),
            gamma_=float(gamma_),
            on="pred_sev",
        )
        cid = f"a1_rank_q{threshold_q}_l{lambda_}_g{gamma_}".replace(".", "")
        scored = tr.score_tail_candidate_multi_split(
            candidate_id=cid,
            base_run_id=base["base_run_id"],
            candidate_family="tail_scale_rank",
            pred_df_all_splits=base["pred_all"],
            transform_kind="tail_rank_scaler",
            transform_params=params,
        )
        candidate_results.append(scored)
        transform_store[cid] = {"transform_kind": "tail_rank_scaler", "transform_params": params}
else:
    print("RUN_PHASE_A1=False")


A1 grid size: 18


## 3) Phase A2 — Tail mapper thresholded (tail-only, monotone)

- Fit uniquement sur la queue des positifs OOF
- Identité stricte sous le seuil
- Extrapolation monotone sûre au-dessus


In [16]:
if RUN_PHASE_A2:
    pos_df = base_primary_oof[(base_primary_oof["y_sev"] > 0) & base_primary_oof["pred_sev"].notna()].copy()
    x_pos = pos_df["pred_sev"].to_numpy(dtype=float)
    y_pos = pos_df["y_sev"].to_numpy(dtype=float)
    grid_a2 = list(itertools.product(THRESHOLD_GRID_A2, MAPPER_MODES_A2, MIN_SAMPLES_TAIL_A2))
    if QUICK_MODE:
        grid_a2 = grid_a2[:6]

    print("A2 grid size:", len(grid_a2))
    for threshold_q, mode, min_tail in grid_a2:
        params = tr.fit_tail_mapper_thresholded(
            x_pos,
            y_pos,
            threshold_q=float(threshold_q),
            mode=str(mode),
            min_samples_tail=int(min_tail),
        )
        cid = f"a2_mapper_q{threshold_q}_{mode}_n{min_tail}".replace(".", "").replace("piecewise_monotone","pwm").replace("isotonic_safe","iso")
        scored = tr.score_tail_candidate_multi_split(
            candidate_id=cid,
            base_run_id=base["base_run_id"],
            candidate_family="tail_mapper_thresholded",
            pred_df_all_splits=base["pred_all"],
            transform_kind="tail_mapper_thresholded",
            transform_params=params,
        )
        candidate_results.append(scored)
        transform_store[cid] = {"transform_kind": "tail_mapper_thresholded", "transform_params": params}
else:
    print("RUN_PHASE_A2=False")


A2 grid size: 6


## 4) Agrégation des candidats tail-only + diagnostics par split


In [17]:
registry_parts = []
dist_parts = []
for item in candidate_results:
    if not item:
        continue
    rr = item.get("registry_rows", pd.DataFrame())
    dd = item.get("dist_df", pd.DataFrame())
    if len(rr):
        registry_parts.append(rr)
    if len(dd):
        dist_parts.append(dd)

tail_candidates_registry = pd.concat(registry_parts, ignore_index=True, sort=False) if registry_parts else pd.DataFrame()
tail_dist_table = pd.concat(dist_parts, ignore_index=True, sort=False) if dist_parts else pd.DataFrame()

if len(tail_candidates_registry):
    tail_candidates_registry = tail_candidates_registry.drop_duplicates(subset=["candidate_id", "split"], keep="last")

tail_diag_by_split = tail_candidates_registry[tail_candidates_registry["split"].astype(str) != "multi"].copy() if len(tail_candidates_registry) else pd.DataFrame()
tail_candidates_multi = tail_candidates_registry[tail_candidates_registry["split"].astype(str) == "multi"].copy() if len(tail_candidates_registry) else pd.DataFrame()

if len(tail_candidates_registry):
    tr._drop_array_cols_for_csv(tail_candidates_registry).to_csv(ARTIFACT_V24 / "tail_candidates_registry.csv", index=False)
    print("saved:", ARTIFACT_V24 / "tail_candidates_registry.csv")
if len(tail_diag_by_split):
    tr._drop_array_cols_for_csv(tail_diag_by_split).to_csv(ARTIFACT_V24 / "tail_diagnostics_by_split.csv", index=False)
    print("saved:", ARTIFACT_V24 / "tail_diagnostics_by_split.csv")

# Save transform params json (all candidates)
jsonable_store = {k: tr._make_jsonable_value(v) for k, v in transform_store.items()}
(ARTIFACT_V24 / "tail_transform_params.json").write_text(json.dumps(jsonable_store, indent=2), encoding="utf-8")
print("saved:", ARTIFACT_V24 / "tail_transform_params.json")

cols_show = [c for c in ['candidate_id','candidate_family','rmse_prime','rmse_gap_secondary','rmse_gap_aux','rmse_split_std','q95_ratio_pos','q99_ratio_pos','rmse_prime_top1pct','body_rmse_proxy','tail_rmse_proxy','distribution_alignment_penalty','selection_score_tail'] if c in tail_candidates_multi.columns]
display(tail_candidates_multi[cols_show].sort_values(['selection_score_tail','rmse_prime'], na_position='last').head(30) if len(tail_candidates_multi) else pd.DataFrame())


saved: c:\Users\icemo\Downloads\Calcul-prime-d-assurance\artifacts\v2_4_tail_recovery\tail_candidates_registry.csv
saved: c:\Users\icemo\Downloads\Calcul-prime-d-assurance\artifacts\v2_4_tail_recovery\tail_diagnostics_by_split.csv
saved: c:\Users\icemo\Downloads\Calcul-prime-d-assurance\artifacts\v2_4_tail_recovery\tail_transform_params.json


,candidate_id,candidate_family,rmse_prime,rmse_gap_secondary,rmse_gap_aux,rmse_split_std,q95_ratio_pos,q99_ratio_pos,rmse_prime_top1pct,body_rmse_proxy,tail_rmse_proxy,distribution_alignment_penalty,selection_score_tail
31,a1_rank_q09_l035_g10,tail_scale_rank,546.188055,-3.682195,-3.412262,1.675807,0.561504,0.443650,4545.178749,105.391932,4545.178749,2.691329,548.974633
63,a1_rank_q095_l01_g30,tail_scale_rank,545.959731,-3.755762,-3.358744,1.684720,0.476975,0.361493,4547.835191,101.232670,4547.835191,1.939834,549.227175
59,a1_rank_q095_l01_g20,tail_scale_rank,545.963263,-3.752221,-3.362095,1.684407,0.476983,0.361493,4547.786303,101.307238,4547.786303,1.941550,549.232423
75,a1_rank_q095_l02_g30,tail_scale_rank,545.999463,-3.777459,-3.392601,1.697287,0.476975,0.394356,4547.444174,101.895635,4547.444174,2.422007,549.256136
43,a1_rank_q09_l05_g10,tail_scale_rank,546.365284,-3.652946,-3.482861,1.683359,0.597731,0.492945,4543.881699,107.703989,4543.881699,2.895113,549.260396
55,a1_rank_q095_l01_g10,tail_scale_rank,545.969804,-3.746498,-3.367886,1.683985,0.477580,0.361493,4547.701055,101.441490,4547.701055,1.964585,549.261999
71,a1_rank_q095_l02_g20,tail_scale_rank,546.007374,-3.769493,-3.400140,1.696613,0.476990,0.394356,4547.346511,102.048198,4547.346511,2.425216,549.267256
27,a1_rank_q09_l02_g30,tail_scale_rank,546.018248,-3.756239,-3.398585,1.692717,0.489359,0.394356,4547.147491,102.327309,4547.147491,2.422694,549.275608
19,a1_rank_q09_l02_g10,tail_scale_rank,546.050660,-3.709100,-3.363791,1.673046,0.525277,0.394356,4546.481539,103.234553,4546.481539,2.414186,549.299512
67,a1_rank_q095_l02_g10,tail_scale_rank,546.022656,-3.755879,-3.413893,1.695689,0.478185,0.394356,4547.176312,102.326108,4547.176312,2.475905,549.333227


## 5) Pareto front (Phase A) + sélection balanced-tail


In [18]:
tail_pareto_front = tr.build_tail_pareto_front(tail_candidates_multi)
if len(tail_pareto_front):
    tr._drop_array_cols_for_csv(tail_pareto_front).to_csv(ARTIFACT_V24 / "tail_pareto_front.csv", index=False)
    print("saved:", ARTIFACT_V24 / "tail_pareto_front.csv")

selected_tail = tr.select_tail_recovery_submissions(tail_candidates_multi, policy="balanced_tail")
display(tail_pareto_front)
display(selected_tail)


saved: c:\Users\icemo\Downloads\Calcul-prime-d-assurance\artifacts\v2_4_tail_recovery\tail_pareto_front.csv


,candidate_id,split,rmse_prime,rmse_prime_top1pct,q95_ratio_pos,q99_ratio_pos,body_rmse_proxy,tail_rmse_proxy,n_oof,n_oof_nonzero,...,selection_score_tail,pred_q90_oof,pred_q99_oof,pred_q90_test,pred_q99_test,q99_test_over_oof,q90_test_over_oof,std_test_over_oof,distribution_alignment_score,pareto_tail_distance
0,baseline_identity,multi,545.929482,4548.227515,0.476975,0.328630,100.615752,4548.227515,NaN,NaN,...,549.394258,158.623141,196.547049,164.156603,178.024449,0.905760,1.034884,0.736300,-1.644221,0.271370
1,a2_mapper_q095_pwm_n150,multi,545.929482,4548.227515,0.476975,0.328630,100.615752,4548.227515,NaN,NaN,...,549.394258,158.623141,196.547049,164.156603,178.024449,0.905760,1.034884,0.736300,-1.644221,0.271370
2,a2_mapper_q095_pwm_n250,multi,545.929482,4548.227515,0.476975,0.328630,100.615752,4548.227515,NaN,NaN,...,549.394258,158.623141,196.547049,164.156603,178.024449,0.905760,1.034884,0.736300,-1.644221,0.271370
3,a1_rank_q095_l01_g30,multi,545.959731,4547.835191,0.476975,0.361493,101.232670,4547.835191,NaN,NaN,...,549.227175,158.623141,202.843672,164.156603,178.024449,0.877644,1.034884,0.729076,-1.939834,0.238507
4,a1_rank_q095_l01_g20,multi,545.963263,4547.786303,0.476983,0.361493,101.307238,4547.786303,NaN,NaN,...,549.232423,158.623141,202.846894,164.156603,178.024449,0.877630,1.034884,0.728288,-1.941550,0.238507
5,a1_rank_q09_l01_g30,multi,545.968011,4547.686701,0.483167,0.361493,101.444275,4547.686701,NaN,NaN,...,549.347987,158.623141,205.476889,164.264848,178.141839,0.866968,1.035567,0.727895,-2.052365,0.238507
6,a1_rank_q09_l01_g20,multi,545.972649,4547.580703,0.489204,0.361493,101.587890,4547.580703,NaN,NaN,...,549.439695,158.623141,208.044183,164.423383,178.650459,0.858714,1.036566,0.728127,-2.139436,0.238507
7,a1_rank_q09_l01_g10,multi,545.981209,4547.353253,0.501126,0.361493,101.887113,4547.353253,NaN,NaN,...,549.504445,159.057794,213.114458,164.423383,181.362789,0.851011,1.033734,0.731465,-2.195626,0.238507
8,a1_rank_q095_l02_g30,multi,545.999463,4547.444174,0.476975,0.394356,101.895635,4547.444174,NaN,NaN,...,549.256136,158.623141,214.222668,164.156603,178.024449,0.831025,1.034884,0.721081,-2.422007,0.205644
9,a1_rank_q095_l02_g20,multi,546.007374,4547.346511,0.476990,0.394356,102.048198,4547.346511,NaN,NaN,...,549.267256,158.623141,214.222668,164.156603,178.024449,0.831025,1.034884,0.719476,-2.425216,0.205644


,candidate_id,split,rmse_prime,rmse_prime_top1pct,q95_ratio_pos,q99_ratio_pos,body_rmse_proxy,tail_rmse_proxy,n_oof,n_oof_nonzero,...,pred_q90_test,pred_q99_test,q99_test_over_oof,q90_test_over_oof,std_test_over_oof,distribution_alignment_score,passes_guardrails,role,risk_tag,tail_strength_rank
0,a1_rank_q09_l035_g10,multi,546.188055,4545.178749,0.561504,0.443650,105.391932,4545.178749,NaN,NaN,...,164.69423,194.956086,0.793084,1.011003,0.716420,-2.691329,True,robust,robust,NaN
1,a2_mapper_q09_pwm_n150,multi,546.502132,4543.569310,0.602693,0.572874,108.739164,4543.569310,NaN,NaN,...,164.69423,208.423836,0.670944,1.001893,0.697284,-3.905453,True,lb_challenger,public_private_risk,-0.572874


## 6) Phase B1 (optionnelle) — Réentraînement sévérité weighted-tail

Par défaut désactivé en quick mode. À activer si Phase A n'améliore pas le compromis.


In [19]:
phase_b_registry = pd.DataFrame()
if RUN_PHASE_B1:
    base_run_row = dict(base["base_run_row"])
    phase_b_parts = []
    for i, sev_override in enumerate(PHASE_B_SEV_PARAM_VARIANTS, start=1):
        dfb = tr.fit_severity_weighted_tail_variant(
            ctx,
            base_run_row=base_run_row,
            seed=SEED,
            feature_set_override=None,
            sev_params_override=sev_override,
            out_dir=ARTIFACT_V24 / f"phase_b_variant_{i}",
        )
        if len(dfb):
            phase_b_parts.append(dfb)
    phase_b_registry = pd.concat(phase_b_parts, ignore_index=True, sort=False) if phase_b_parts else pd.DataFrame()
    display(phase_b_registry)
else:
    print("RUN_PHASE_B1=False (default)")


,run_id,feature_set,engine,family,tweedie_power,config_id,seed,severity_mode,calibration,tail_mapper,...,tail_penalty,rmse_gap_secondary_pos,rmse_gap_aux_pos,shakeup_penalty,selection_score_quick,passes_local_guardrails,candidate_family,base_run_id,candidate_id,selection_score_tail
0,base_v2|catboost|two_part_tweedie|1.5|cb_v2_c1...,base_v2,catboost,two_part_tweedie,1.5,cb_v2_c1,42,weighted_tail,isotonic,piecewise_monotone_safe,...,6.957555,0.000000,0.000000,0.0,570.943847,True,sev_retrain_weighted_tail,base_v2|catboost|two_part_tweedie|cb_v2_c1|42|...,sev_retrain_weighted_tail,570.943847
1,compact_v2|catboost|two_part_tweedie|1.5|cb_v2...,compact_v2,catboost,two_part_tweedie,1.5,cb_v2_c1,42,weighted_tail,isotonic,piecewise_monotone_safe,...,3.313505,4.284645,0.000000,0.0,570.000954,False,sev_retrain_weighted_tail,base_v2|catboost|two_part_tweedie|cb_v2_c1|42|...,sev_retrain_weighted_tail,570.000954
2,robust_v2|catboost|two_part_tweedie|1.5|cb_v2_...,robust_v2,catboost,two_part_tweedie,1.5,cb_v2_c1,42,weighted_tail,isotonic,piecewise_monotone_safe,...,6.186399,0.000000,0.358092,0.0,572.655457,True,sev_retrain_weighted_tail,base_v2|catboost|two_part_tweedie|cb_v2_c1|42|...,sev_retrain_weighted_tail,572.655457


## 7) Sélection finale V2.4 + génération des deux soumissions

- `robust`: meilleur compromis RMSE/queue/stabilité
- `lb_challenger`: candidat plus agressif (risque public/private explicite)


In [20]:
# Pool de sélection: Phase A seulement par défaut (Phase B optionnel et non sélectionnable par défaut)
selection_pool = tail_candidates_multi.copy()
if RUN_PHASE_B1 and ALLOW_PHASE_B_SELECTION and len(phase_b_registry):
    selection_pool = pd.concat([selection_pool, phase_b_registry], ignore_index=True, sort=False)

selected_tail = tr.select_tail_recovery_submissions(selection_pool, policy="balanced_tail")
display(selected_tail)

outputs_v24 = tr.materialize_tail_recovery_submissions(
    ctx,
    base_run_row=base["base_run_row"],
    selected_df=selected_tail,
    transform_store=transform_store,
    out_dir=ARTIFACT_V24,
)
print("submission_paths:", outputs_v24.get("submission_paths", {}))
display(outputs_v24.get("pred_audits", pd.DataFrame()))


,candidate_id,split,rmse_prime,rmse_prime_top1pct,q95_ratio_pos,q99_ratio_pos,body_rmse_proxy,tail_rmse_proxy,n_oof,n_oof_nonzero,...,pred_q90_test,pred_q99_test,q99_test_over_oof,q90_test_over_oof,std_test_over_oof,distribution_alignment_score,passes_guardrails,role,risk_tag,tail_strength_rank
0,a1_rank_q09_l035_g10,multi,546.188055,4545.178749,0.561504,0.443650,105.391932,4545.178749,NaN,NaN,...,164.69423,194.956086,0.793084,1.011003,0.716420,-2.691329,True,robust,robust,NaN
1,a2_mapper_q09_pwm_n150,multi,546.502132,4543.569310,0.602693,0.572874,108.739164,4543.569310,NaN,NaN,...,164.69423,208.423836,0.670944,1.001893,0.697284,-3.905453,True,lb_challenger,public_private_risk,-0.572874


submission_paths: {'robust': WindowsPath('c:/Users/icemo/Downloads/Calcul-prime-d-assurance/artifacts/v2_4_tail_recovery/submission_v2_4_robust.csv'), 'lb_challenger': WindowsPath('c:/Users/icemo/Downloads/Calcul-prime-d-assurance/artifacts/v2_4_tail_recovery/submission_v2_4_lb_challenger.csv')}


,role,candidate_id,run_id,split,sample,n,pred_mean,pred_std,pred_q50,pred_q90,...,pred_share_nonzero,pred_identical_share,pred_identical_share_nonzero,pred_q99_q90_ratio,pred_n_nonzero,pred_q90_nonzero,pred_q99_nonzero,pred_q99_q90_ratio_nonzero,collapse_use_nonzero_support,distribution_collapse_flag
0,robust,a1_rank_q09_l035_g10,a1_rank_q09_l035_g10,test,test,50000,98.290521,169.888724,52.124974,235.670510,...,1.0,0.02328,0.02328,3.625657,50000,235.670510,854.460356,3.625657,0,0
1,lb_challenger,a2_mapper_q09_pwm_n150,a2_mapper_q09_pwm_n150,test,test,50000,226.818510,657.416537,52.147584,541.565162,...,1.0,0.02328,0.02328,6.158605,50000,541.565162,3335.286092,6.158605,0,0


In [21]:
# Rapport de décision V2.4
decision_path = tr.write_tail_decision_report(
    base_info=base,
    candidates_df=tail_candidates_registry,
    pareto_df=tail_pareto_front,
    selected_df=selected_tail,
    out_dir=ARTIFACT_V24,
)
print("saved:", decision_path)
print(decision_path.read_text(encoding="utf-8")[:5000])


saved: c:\Users\icemo\Downloads\Calcul-prime-d-assurance\artifacts\v2_4_tail_recovery\submission_decision_v2_4_tail.md
# Submission decision V2.4 Tail Recovery

## 1) Contexte
- Base run V2 (ancre robuste): `base_v2|catboost|two_part_tweedie|cb_v2_c1|42|weighted_tail|isotonic|isotonic`
- Objectif: corriger la queue (q95/q99) sans casser RMSE global ni la stabilité inter-splits.

## 2) Résumé candidats

| candidate_id            | candidate_family        |   rmse_prime |   rmse_gap_secondary |   rmse_gap_aux |   q95_ratio_pos |   q99_ratio_pos |   rmse_prime_top1pct |   selection_score_tail |
|:------------------------|:------------------------|-------------:|---------------------:|---------------:|----------------:|----------------:|---------------------:|-----------------------:|
| a1_rank_q09_l035_g10    | tail_scale_rank         |      546.188 |             -3.68219 |       -3.41226 |        0.561504 |        0.44365  |              4545.18 |                548.975 |
| a1_rank_q095_

## 8) Checks obligatoires (sanity / robustesse / format)


In [22]:
checks = []

# Transform checks (identity under threshold / monotonicity) on selected transforms
for _, row in selected_tail.iterrows() if len(selected_tail) else []:
    cid = str(row.get("candidate_id"))
    payload = transform_store.get(cid, {})
    kind = payload.get("transform_kind")
    params = payload.get("transform_params", {})
    if kind == "tail_rank_scaler":
        ref = np.asarray(params.get("rank_ref_sorted", []), dtype=float)
        if len(ref):
            grid = np.linspace(0, np.quantile(ref, 0.999), 200)
            out = tr.apply_tail_rank_scaler(grid, rank_ref=ref, params=params)
            checks.append({"check":"monotonic_tail_rank", "candidate_id":cid, "ok": bool(np.all(np.diff(out) >= -1e-9))})
            thr = float(params.get("threshold_val", 0.0))
            below = grid[grid <= thr]
            if len(below):
                out_below = tr.apply_tail_rank_scaler(below, rank_ref=ref, params=params)
                checks.append({"check":"identity_below_threshold_tail_rank", "candidate_id":cid, "ok": bool(np.allclose(out_below, below))})
    elif kind == "tail_mapper_thresholded":
        x = np.linspace(0, 5000, 300)
        out = tr.apply_tail_mapper_thresholded(x, params)
        checks.append({"check":"monotonic_tail_mapper_thresholded", "candidate_id":cid, "ok": bool(np.all(np.diff(out) >= -1e-9))})

# Candidate metrics guardrails
if len(selected_tail):
    for _, r in selected_tail.iterrows():
        checks.append({"check":"q99_not_too_low", "candidate_id":str(r.get("candidate_id")), "ok": bool(pd.to_numeric(pd.Series([r.get('q99_ratio_pos')]), errors='coerce').fillna(0).iloc[0] >= 0.10 or pd.isna(r.get('q99_ratio_pos')) )})
        checks.append({"check":"q99_not_overcorrected", "candidate_id":str(r.get("candidate_id")), "ok": bool(pd.isna(r.get('q99_ratio_pos')) or float(r.get('q99_ratio_pos')) <= 0.85)})

# Submission checks
for role, path in (outputs_v24.get("submission_paths", {}) or {}).items():
    p = Path(path)
    ok = p.exists()
    if ok:
        sub = pd.read_csv(p)
        pred_col = "pred" if "pred" in sub.columns else None
        checks.append({"check":"submission_exists", "candidate_id":role, "ok": True})
        checks.append({"check":"submission_n_rows_50000", "candidate_id":role, "ok": bool(len(sub) == 50000)})
        checks.append({"check":"submission_cols", "candidate_id":role, "ok": bool(list(sub.columns) == ['index','pred'])})
        if pred_col:
            checks.append({"check":"submission_pred_nonneg", "candidate_id":role, "ok": bool((pd.to_numeric(sub[pred_col], errors='coerce').fillna(-1) >= 0).all())})
    else:
        checks.append({"check":"submission_exists", "candidate_id":role, "ok": False})

checks_df = pd.DataFrame(checks)
display(checks_df)
print("all checks passed:", bool(checks_df['ok'].all()) if len(checks_df) else True)


,check,candidate_id,ok
0,monotonic_tail_rank,a1_rank_q09_l035_g10,True
1,identity_below_threshold_tail_rank,a1_rank_q09_l035_g10,True
2,monotonic_tail_mapper_thresholded,a2_mapper_q09_pwm_n150,True
3,q99_not_too_low,a1_rank_q09_l035_g10,True
4,q99_not_overcorrected,a1_rank_q09_l035_g10,True
5,q99_not_too_low,a2_mapper_q09_pwm_n150,True
6,q99_not_overcorrected,a2_mapper_q09_pwm_n150,True
7,submission_exists,robust,True
8,submission_n_rows_50000,robust,True
9,submission_cols,robust,True


all checks passed: True


## Artefacts attendus V2.4

- `tail_candidates_registry.csv`
- `tail_pareto_front.csv`
- `tail_diagnostics_by_split.csv`
- `tail_transform_params.json`
- `submission_v2_4_robust.csv`
- `submission_v2_4_lb_challenger.csv`
- `submission_decision_v2_4_tail.md`
